# 🏏 IPL Prediction — Complete GPU Training Pipeline

**What's new in this notebook vs the basic one:**

| Feature | Basic notebook | This notebook |
|---------|---------------|---------------|
| LightGBM | ❌ | ✅ GPU-accelerated |
| Hyperparameter tuning | manual | ✅ Optuna (100+ trials) |
| ELO ratings | ❌ | ✅ chronological team ranking |
| Extended form windows | 1 (5-match) | ✅ 3 / 5 / 10 / 20-match |
| Phase-specific models | ❌ | ✅ powerplay / middle / death |
| Deep learning arch | entity embedding MLP | ✅ TabTransformer (attention) |
| Ensemble method | best-of selection | ✅ 2-level stacking meta-learner |
| Match predictions | ❌ | ✅ one-by-one for all upcoming |
| Tournament simulation | ❌ | ✅ 10,000-run Monte Carlo |

**Runtime: T4 GPU required** — `Runtime → Change runtime type → T4 GPU`

---

## ⚙️ Step 0 — Config

In [ ]:
# ─── EDIT THESE ───────────────────────────────────────────────────────────
GITHUB_REPO_URL  = ""         # your repo URL, e.g. https://github.com/soulrahulrk/ipl-prediction.git
GITHUB_BRANCH    = "main"
USE_GOOGLE_DRIVE = True       # mount Drive to save models
DRIVE_SAVE_PATH  = "/content/drive/MyDrive/ipl_models_advanced"
FORCE_REPROCESS  = False      # re-run preprocess even if ipl_features.csv exists
OPTUNA_TRIALS    = 60         # HPO trials per model (60 = ~15min per model)
SKIP_DL          = False      # skip TabTransformer training
SKIP_STACKING    = False      # skip stacking ensemble (takes extra time)
# ──────────────────────────────────────────────────────────────────────────
print("Config loaded.")

## 📦 Step 1 — Install Dependencies

In [ ]:
%%time
!pip install -q \
    'numpy>=1.26,<3' 'pandas>=2.2,<3' 'scikit-learn>=1.7,<1.8' \
    'xgboost>=2.1,<3' 'catboost>=1.2,<2' 'lightgbm>=4.3' \
    'optuna>=3.6' 'optuna-integration[sklearn]' \
    'joblib>=1.4' 'matplotlib>=3.9' 'seaborn' 'tqdm'

# LightGBM GPU check
import lightgbm as lgb, torch
GPU_AVAIL = torch.cuda.is_available()
if GPU_AVAIL:
    print(f'\n✅ GPU: {torch.cuda.get_device_name(0)}  CUDA {torch.version.cuda}')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU — some models will run on CPU (slower)')
    DEVICE = 'cpu'
print(f'LightGBM {lgb.__version__}  |  PyTorch {torch.__version__}')

## 💾 Step 2 — Mount Google Drive

In [ ]:
import os
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    print(f'✅ Drive mounted → {DRIVE_SAVE_PATH}')
else:
    print('Drive not mounted.')

## 📂 Step 3 — Get Code & Data

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/ipl_prediction')

if GITHUB_REPO_URL:
    if PROJECT_DIR.exists():
        !cd /content/ipl_prediction && git pull
    else:
        !git clone --depth 1 --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} /content/ipl_prediction
    print('✅ Repo ready.')
else:
    print('⚠️  Set GITHUB_REPO_URL above, or manually upload your project.')
    print('   Then set PROJECT_DIR = Path("/content/your-folder-name")')

# Add both project root and colab_training to path
for p in [str(PROJECT_DIR), str(PROJECT_DIR / 'colab_training')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify imports
from ipl_predictor import CATEGORICAL_FEATURES, NUMERIC_FEATURES, ACTIVE_IPL_TEAMS_2026
from feature_engine import build_advanced_features, ADVANCED_NUMERIC_EXTRAS
from model_zoo import StackingRegressor, StackingClassifier, TorchTabTransformerRegressor, TorchTabTransformerClassifier
from tournament import MatchPredictor, TournamentSimulator, UPCOMING_2026, predict_all_upcoming
print('✅ All imports OK.')
print(f'   Base features : {len(CATEGORICAL_FEATURES)} cat + {len(NUMERIC_FEATURES)} num')
print(f'   Extra features: {len(ADVANCED_NUMERIC_EXTRAS)} new numeric columns')

In [ ]:
%%time
import io, urllib.request, zipfile
RAW_DIR = PROJECT_DIR / 'data' / 'ipl_csv2'
RAW_DIR.mkdir(parents=True, exist_ok=True)

existing = list(RAW_DIR.glob('*.csv'))
if existing and not FORCE_REPROCESS:
    print(f'✅ {len(existing)} CSV files already present.')
else:
    print('Downloading IPL CSV2 from Cricsheet ...')
    req = urllib.request.Request(
        'https://cricsheet.org/downloads/ipl_csv2.zip',
        headers={'User-Agent': 'Mozilla/5.0', 'Accept': '*/*'}
    )
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(req, timeout=180) as r:
                payload = r.read()
            print(f'   Downloaded {len(payload)/1e6:.1f} MB')
            break
        except Exception as e:
            print(f'   Attempt {attempt} failed: {e}')
            import time; time.sleep(5)
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        zf.extractall(RAW_DIR)
    print(f'✅ Extracted {len(list(RAW_DIR.glob("*.csv")))} files.')

In [ ]:
%%time
FEATURES_PATH = PROJECT_DIR / 'data' / 'processed' / 'ipl_features.csv'

if FEATURES_PATH.exists() and not FORCE_REPROCESS:
    print(f'✅ ipl_features.csv exists ({FEATURES_PATH.stat().st_size/1e6:.1f} MB). Skipping preprocess.')
else:
    print('Running preprocessing (~10-20 min) ...')
    !cd {PROJECT_DIR} && python scripts/preprocess_ipl.py

## 🔧 Step 4 — Load & Add Advanced Features

In [ ]:
%%time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from ipl_predictor import season_to_year, ACTIVE_IPL_TEAMS_2026

print('Loading ipl_features.csv ...')
df_raw = pd.read_csv(FEATURES_PATH, low_memory=False)
df_raw['season_int'] = df_raw['season'].apply(season_to_year)

# Active teams only
mask_active = (
    df_raw['batting_team'].isin(ACTIVE_IPL_TEAMS_2026) &
    df_raw['bowling_team'].isin(ACTIVE_IPL_TEAMS_2026)
)
df_base = df_raw[mask_active & (df_raw['season_int'] >= 2008)].copy()
print(f'Base rows : {len(df_base):,}  |  seasons: {sorted(df_base["season_int"].dropna().unique().astype(int))}')

In [ ]:
%%time
from feature_engine import EloSystem, build_advanced_features

# Build all advanced features + fit ELO
df, elo = build_advanced_features(df_base, verbose=True)

print('\n=== Current ELO Standings ===')
print(elo.ratings_df().to_string(index=False))

## 📊 Step 5 — EDA & Feature Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ELO distribution
elo_df = elo.ratings_df()
axes[0].barh(elo_df['team'].str.split().str[-1], elo_df['elo'], color='steelblue')
axes[0].axvline(1500, color='red', linestyle='--', alpha=0.6, label='Baseline')
axes[0].set_title('Current ELO Ratings')
axes[0].set_xlabel('ELO')
axes[0].legend()

# ELO vs win-rate scatter per season
wins = df[df['win'].notna()].groupby(['batting_team', 'season_int'])['win'].mean().reset_index()
elo_latest = elo_df.set_index('team')['elo']
wins['elo'] = wins['batting_team'].map(elo_latest)
axes[1].scatter(wins['elo'], wins['win'], alpha=0.4, s=20, color='coral')
axes[1].set_title('Team ELO vs Win Rate (per season)')
axes[1].set_xlabel('ELO'); axes[1].set_ylabel('Win Rate')

# Phase run-rates
phases = df.groupby('phase')[['bat_pp_rr','bat_mid_rr','bat_death_rr']].mean().T
phases.plot(kind='bar', ax=axes[2])
axes[2].set_title('Phase Run-Rates')
axes[2].set_ylabel('Avg Runs/Ball')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## ✂️ Step 6 — Feature Set & Train / Val / Test Splits

In [ ]:
from ipl_predictor import CATEGORICAL_FEATURES, NUMERIC_FEATURES
from feature_engine import ADVANCED_NUMERIC_EXTRAS

CAT_FEATS  = CATEGORICAL_FEATURES
NUM_FEATS  = NUMERIC_FEATURES + [c for c in ADVANCED_NUMERIC_EXTRAS if c in df.columns]
ALL_FEATS  = CAT_FEATS + NUM_FEATS

# Remove columns not present
ALL_FEATS = [c for c in ALL_FEATS if c in df.columns]
print(f'Total features: {len(ALL_FEATS)}  ({len(CAT_FEATS)} cat + {len([f for f in ALL_FEATS if f not in CAT_FEATS])} num)')

# Train / Valid / Test splits (same logic as train_all_models.py)
seasons = sorted(df['season_int'].dropna().unique().astype(int))
test_yr, valid_yr, train_yrs = seasons[-1], seasons[-2], seasons[:-2]
print(f'Train seasons: {train_yrs}')
print(f'Valid season : {valid_yr}')
print(f'Test season  : {test_yr}  (hold-out)')

train = df[df['season_int'].isin(train_yrs)].copy()
valid = df[df['season_int'] == valid_yr].copy()
test  = df[df['season_int'] == test_yr].copy()
tv    = pd.concat([train, valid])
print(f'Train: {len(train):,}  Valid: {len(valid):,}  Test: {len(test):,}')

In [ ]:
# Score task: balls_left > 0, total_runs known
tr_s = train[(train['balls_left'] > 0) & train['total_runs'].notna()]
va_s = valid[(valid['balls_left'] > 0) & valid['total_runs'].notna()]
te_s = test[ (test ['balls_left'] > 0) & test ['total_runs'].notna()]
tv_s = pd.concat([tr_s, va_s])

X_tr_s, y_tr_s = tr_s[ALL_FEATS], tr_s['total_runs']
X_va_s, y_va_s = va_s[ALL_FEATS], va_s['total_runs']
X_te_s, y_te_s = te_s[ALL_FEATS], te_s['total_runs']
X_tv_s, y_tv_s = tv_s[ALL_FEATS], tv_s['total_runs']

# Win task
tr_w = train[(train['balls_left'] > 0) & train['win'].notna()]
va_w = valid[(valid['balls_left'] > 0) & valid['win'].notna()]
te_w = test[ (test ['balls_left'] > 0) & test ['win'].notna()]
tv_w = pd.concat([tr_w, va_w])

X_tr_w, y_tr_w = tr_w[ALL_FEATS], tr_w['win'].astype(int)
X_va_w, y_va_w = va_w[ALL_FEATS], va_w['win'].astype(int)
X_te_w, y_te_w = te_w[ALL_FEATS], te_w['win'].astype(int)
X_tv_w, y_tv_w = tv_w[ALL_FEATS], tv_w['win'].astype(int)

print(f'Score task — Train: {len(X_tr_s):,}  Val: {len(X_va_s):,}  Test: {len(X_te_s):,}')
print(f'Win   task — Train: {len(X_tr_w):,}  Val: {len(X_va_w):,}  Test: {len(X_te_w):,}')

## ⚡ Step 7 — GPU Baseline: HGB (fast benchmark)

In [ ]:
%%time
import joblib, time
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, log_loss, accuracy_score, brier_score_loss

ADV_MODELS_DIR = PROJECT_DIR / 'models' / 'advanced'
ADV_MODELS_DIR.mkdir(parents=True, exist_ok=True)

def hgb_preprocessor():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='constant', fill_value=0),  [f for f in ALL_FEATS if f not in CAT_FEATS]),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FEATS),
    ])

NUM_FEATS_ONLY = [f for f in ALL_FEATS if f not in CAT_FEATS]

t0 = time.time()
hgb_score = Pipeline([
    ('pre', hgb_preprocessor()),
    ('m',   HistGradientBoostingRegressor(
        max_iter=500, max_depth=8, learning_rate=0.07,
        l2_regularization=0.1, random_state=42)),
])
hgb_score.fit(X_tr_s, y_tr_s)
hgb_s_mae  = mean_absolute_error(y_te_s, hgb_score.predict(X_te_s))
hgb_s_rmse = mean_squared_error(y_te_s, hgb_score.predict(X_te_s))**0.5
print(f'[HGB Score] MAE={hgb_s_mae:.3f}  RMSE={hgb_s_rmse:.3f}  ({time.time()-t0:.0f}s)')
joblib.dump(hgb_score, ADV_MODELS_DIR / 'score_hgb_adv.pkl')

t0 = time.time()
hgb_win_base = Pipeline([
    ('pre', hgb_preprocessor()),
    ('m',   HistGradientBoostingClassifier(
        max_iter=500, max_depth=6, learning_rate=0.07,
        l2_regularization=0.1, random_state=42)),
])
hgb_win = CalibratedClassifierCV(hgb_win_base, method='isotonic', cv=5)
hgb_win.fit(X_tv_w, y_tv_w)
hgb_w_probs = hgb_win.predict_proba(X_te_w)[:, 1]
hgb_w_acc   = accuracy_score(y_te_w, (hgb_w_probs>=0.5).astype(int))
hgb_w_ll    = log_loss(y_te_w, hgb_w_probs)
print(f'[HGB Win]   Acc={hgb_w_acc:.4f}  LogLoss={hgb_w_ll:.4f}  ({time.time()-t0:.0f}s)')
joblib.dump(hgb_win, ADV_MODELS_DIR / 'win_hgb_adv.pkl')

## ⚡ Step 8 — GPU Models: XGBoost

In [ ]:
%%time
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder

def tree_preprocessor():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='constant', fill_value=0),  NUM_FEATS_ONLY),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)),
        ]), CAT_FEATS),
    ])

xgb_device = 'cuda' if GPU_AVAIL else 'cpu'
pre_xgb = tree_preprocessor()
X_tr_xgb = pre_xgb.fit_transform(X_tr_s)
X_va_xgb = pre_xgb.transform(X_va_s)
X_te_xgb = pre_xgb.transform(X_te_s)
X_tv_xgb = pre_xgb.transform(X_tv_s)

t0 = time.time()
xgb_score = xgb.XGBRegressor(
    n_estimators=800, max_depth=7, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5, reg_alpha=0.1,
    device=xgb_device, tree_method='hist',
    early_stopping_rounds=40, eval_metric='rmse', random_state=42, n_jobs=-1,
)
xgb_score.fit(X_tr_xgb, y_tr_s, eval_set=[(X_va_xgb, y_va_s)], verbose=False)
# Refit on train+valid with best iteration
xgb_score_final = xgb.XGBRegressor(
    n_estimators=xgb_score.best_iteration+1, max_depth=7, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5, reg_alpha=0.1,
    device=xgb_device, tree_method='hist', random_state=42, n_jobs=-1,
)
xgb_score_final.fit(X_tv_xgb, y_tv_s)
xgb_s_mae  = mean_absolute_error(y_te_s, xgb_score_final.predict(X_te_xgb))
xgb_s_rmse = mean_squared_error(y_te_s, xgb_score_final.predict(X_te_xgb))**0.5
print(f'[XGB Score] MAE={xgb_s_mae:.3f}  RMSE={xgb_s_rmse:.3f}  best_iter={xgb_score.best_iteration}  ({time.time()-t0:.0f}s)')
joblib.dump({'pre': pre_xgb, 'model': xgb_score_final}, ADV_MODELS_DIR / 'score_xgb_adv.pkl')

In [ ]:
%%time
pre_xgb_w = tree_preprocessor()
X_tr_xgb_w = pre_xgb_w.fit_transform(X_tr_w)
X_va_xgb_w = pre_xgb_w.transform(X_va_w)
X_te_xgb_w = pre_xgb_w.transform(X_te_w)
X_tv_xgb_w = pre_xgb_w.transform(X_tv_w)

t0 = time.time()
xgb_win = xgb.XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5,
    device=xgb_device, tree_method='hist',
    early_stopping_rounds=40, eval_metric='logloss', random_state=42, n_jobs=-1,
)
xgb_win.fit(X_tr_xgb_w, y_tr_w, eval_set=[(X_va_xgb_w, y_va_w)], verbose=False)
xgb_win_final = xgb.XGBClassifier(
    n_estimators=xgb_win.best_iteration+1, max_depth=6, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5,
    device=xgb_device, tree_method='hist', random_state=42, n_jobs=-1,
)
xgb_win_final.fit(X_tv_xgb_w, y_tv_w)
xgb_win_cal = CalibratedClassifierCV(xgb_win_final, method='isotonic', cv=5)
xgb_win_cal.fit(X_tv_xgb_w, y_tv_w)
xgb_w_probs = xgb_win_cal.predict_proba(X_te_xgb_w)[:, 1]
xgb_w_acc   = accuracy_score(y_te_w, (xgb_w_probs>=0.5).astype(int))
xgb_w_ll    = log_loss(y_te_w, xgb_w_probs)
print(f'[XGB Win]   Acc={xgb_w_acc:.4f}  LogLoss={xgb_w_ll:.4f}  ({time.time()-t0:.0f}s)')
joblib.dump({'pre': pre_xgb_w, 'model': xgb_win_cal}, ADV_MODELS_DIR / 'win_xgb_adv.pkl')

## ⚡ Step 9 — GPU Models: LightGBM (NEW)

In [ ]:
%%time
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV

lgb_device = 'gpu' if GPU_AVAIL else 'cpu'
print(f'LightGBM device: {lgb_device}')

# Ordinal encoding for LightGBM (supports categorical features natively)
def lgb_preprocessor():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='constant', fill_value=0),  NUM_FEATS_ONLY),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), CAT_FEATS),
    ])

pre_lgb = lgb_preprocessor()
X_tr_lgb = pre_lgb.fit_transform(X_tr_s)
X_va_lgb = pre_lgb.transform(X_va_s)
X_te_lgb = pre_lgb.transform(X_te_s)
X_tv_lgb = pre_lgb.transform(X_tv_s)

t0 = time.time()
lgb_score = lgb.LGBMRegressor(
    n_estimators=1000, max_depth=8, num_leaves=127,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.5, reg_alpha=0.1, min_child_samples=20,
    device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_score.fit(
    X_tr_lgb, y_tr_s,
    eval_set=[(X_va_lgb, y_va_s)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)
# Refit on train+valid
lgb_score_final = lgb.LGBMRegressor(
    n_estimators=lgb_score.best_iteration_, max_depth=8, num_leaves=127,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.5, reg_alpha=0.1, min_child_samples=20,
    device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_score_final.fit(X_tv_lgb, y_tv_s)
lgb_s_mae  = mean_absolute_error(y_te_s, lgb_score_final.predict(X_te_lgb))
lgb_s_rmse = mean_squared_error(y_te_s, lgb_score_final.predict(X_te_lgb))**0.5
print(f'[LGB Score] MAE={lgb_s_mae:.3f}  RMSE={lgb_s_rmse:.3f}  best_iter={lgb_score.best_iteration_}  ({time.time()-t0:.0f}s)')
joblib.dump({'pre': pre_lgb, 'model': lgb_score_final}, ADV_MODELS_DIR / 'score_lgb_adv.pkl')

In [ ]:
%%time
pre_lgb_w = lgb_preprocessor()
X_tr_lgb_w = pre_lgb_w.fit_transform(X_tr_w)
X_va_lgb_w = pre_lgb_w.transform(X_va_w)
X_te_lgb_w = pre_lgb_w.transform(X_te_w)
X_tv_lgb_w = pre_lgb_w.transform(X_tv_w)

t0 = time.time()
lgb_win = lgb.LGBMClassifier(
    n_estimators=1000, max_depth=7, num_leaves=63,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.5, reg_alpha=0.1, min_child_samples=20,
    device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_win.fit(
    X_tr_lgb_w, y_tr_w,
    eval_set=[(X_va_lgb_w, y_va_w)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)
lgb_win_final = lgb.LGBMClassifier(
    n_estimators=lgb_win.best_iteration_, max_depth=7, num_leaves=63,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.5, reg_alpha=0.1, min_child_samples=20,
    device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_win_final.fit(X_tv_lgb_w, y_tv_w)
lgb_win_cal = CalibratedClassifierCV(lgb_win_final, method='isotonic', cv=5)
lgb_win_cal.fit(X_tv_lgb_w, y_tv_w)
lgb_w_probs = lgb_win_cal.predict_proba(X_te_lgb_w)[:, 1]
lgb_w_acc   = accuracy_score(y_te_w, (lgb_w_probs>=0.5).astype(int))
lgb_w_ll    = log_loss(y_te_w, lgb_w_probs)
print(f'[LGB Win]   Acc={lgb_w_acc:.4f}  LogLoss={lgb_w_ll:.4f}  ({time.time()-t0:.0f}s)')
joblib.dump({'pre': pre_lgb_w, 'model': lgb_win_cal}, ADV_MODELS_DIR / 'win_lgb_adv.pkl')

## ⚡ Step 10 — GPU Models: CatBoost

In [ ]:
%%time
from catboost import CatBoostRegressor, CatBoostClassifier

cb_task = 'GPU' if GPU_AVAIL else 'CPU'

def cb_prep(X): 
    out = X.copy()
    out[CAT_FEATS] = out[CAT_FEATS].fillna('Unknown').astype(str)
    return out

t0 = time.time()
cb_score = CatBoostRegressor(
    iterations=1200, depth=8, learning_rate=0.06,
    l2_leaf_reg=2, cat_features=CAT_FEATS,
    task_type=cb_task, eval_metric='RMSE',
    early_stopping_rounds=50, verbose=0, random_seed=42,
)
cb_score.fit(cb_prep(X_tr_s), y_tr_s, eval_set=(cb_prep(X_va_s), y_va_s))
best_cb_s = cb_score.best_iteration_ or 1200
cb_score_final = CatBoostRegressor(
    iterations=best_cb_s, depth=8, learning_rate=0.06,
    l2_leaf_reg=2, cat_features=CAT_FEATS,
    task_type=cb_task, verbose=0, random_seed=42,
)
cb_score_final.fit(cb_prep(X_tv_s), y_tv_s)
cb_s_mae  = mean_absolute_error(y_te_s, cb_score_final.predict(cb_prep(X_te_s)))
cb_s_rmse = mean_squared_error(y_te_s, cb_score_final.predict(cb_prep(X_te_s)))**0.5
print(f'[CB  Score] MAE={cb_s_mae:.3f}  RMSE={cb_s_rmse:.3f}  ({time.time()-t0:.0f}s)')
joblib.dump(cb_score_final, ADV_MODELS_DIR / 'score_cb_adv.pkl')

In [ ]:
%%time
t0 = time.time()
cb_win = CatBoostClassifier(
    iterations=1200, depth=7, learning_rate=0.06,
    l2_leaf_reg=2, cat_features=CAT_FEATS,
    task_type=cb_task, eval_metric='Logloss',
    early_stopping_rounds=50, verbose=0, random_seed=42,
)
cb_win.fit(cb_prep(X_tr_w), y_tr_w, eval_set=(cb_prep(X_va_w), y_va_w))
best_cb_w = cb_win.best_iteration_ or 1200
cb_win_final = CatBoostClassifier(
    iterations=best_cb_w, depth=7, learning_rate=0.06,
    l2_leaf_reg=2, cat_features=CAT_FEATS,
    task_type=cb_task, verbose=0, random_seed=42,
)
cb_win_final.fit(cb_prep(X_tv_w), y_tv_w)
cb_win_cal = CalibratedClassifierCV(cb_win_final, method='isotonic', cv=5)
cb_win_cal.fit(cb_prep(X_tv_w), y_tv_w)
cb_w_probs = cb_win_cal.predict_proba(cb_prep(X_te_w))[:, 1]
cb_w_acc   = accuracy_score(y_te_w, (cb_w_probs>=0.5).astype(int))
cb_w_ll    = log_loss(y_te_w, cb_w_probs)
print(f'[CB  Win]   Acc={cb_w_acc:.4f}  LogLoss={cb_w_ll:.4f}  ({time.time()-t0:.0f}s)')
joblib.dump(cb_win_cal, ADV_MODELS_DIR / 'win_cb_adv.pkl')

## 🔬 Step 11 — Optuna Hyperparameter Optimization (NEW)

In [ ]:
%%time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'Running Optuna HPO: {OPTUNA_TRIALS} trials each for XGBoost & LightGBM score models ...')

# ── XGBoost Score HPO ──────────────────────────────────────────────────────
def xgb_score_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 300, 1500),
        max_depth         = trial.suggest_int('max_depth', 4, 10),
        learning_rate     = trial.suggest_float('lr', 0.01, 0.3, log=True),
        subsample         = trial.suggest_float('sub', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('col', 0.5, 1.0),
        reg_lambda        = trial.suggest_float('lam', 0.1, 10.0, log=True),
        reg_alpha         = trial.suggest_float('alpha', 0.0, 5.0),
        min_child_weight  = trial.suggest_int('mcw', 1, 10),
        gamma             = trial.suggest_float('gamma', 0.0, 1.0),
    )
    m = xgb.XGBRegressor(
        **params, device=xgb_device, tree_method='hist', random_state=42, n_jobs=-1
    )
    m.fit(X_tr_xgb, y_tr_s, eval_set=[(X_va_xgb, y_va_s)],
          early_stopping_rounds=30, verbose=False)
    preds = m.predict(X_va_xgb)
    return mean_absolute_error(y_va_s, preds)

study_xgb_s = optuna.create_study(direction='minimize', study_name='xgb_score')
study_xgb_s.optimize(xgb_score_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb_s.best_params
print(f'\nXGB best val MAE: {study_xgb_s.best_value:.4f}')
print(f'Best params: {best_xgb_params}')

In [ ]:
%%time
# Retrain XGBoost with tuned params on train+valid, evaluate on test
best_p = study_xgb_s.best_params
xgb_tuned = xgb.XGBRegressor(
    n_estimators=best_p['n_estimators'],
    max_depth=best_p['max_depth'],
    learning_rate=best_p['lr'],
    subsample=best_p['sub'],
    colsample_bytree=best_p['col'],
    reg_lambda=best_p['lam'],
    reg_alpha=best_p['alpha'],
    min_child_weight=best_p['mcw'],
    gamma=best_p['gamma'],
    device=xgb_device, tree_method='hist', random_state=42, n_jobs=-1,
)
xgb_tuned.fit(X_tv_xgb, y_tv_s)
xgb_tuned_mae  = mean_absolute_error(y_te_s, xgb_tuned.predict(X_te_xgb))
xgb_tuned_rmse = mean_squared_error(y_te_s, xgb_tuned.predict(X_te_xgb))**0.5
print(f'[XGB Tuned Score] MAE={xgb_tuned_mae:.3f}  RMSE={xgb_tuned_rmse:.3f}')
print(f'  Improvement over baseline: MAE {xgb_s_mae - xgb_tuned_mae:+.3f}  RMSE {xgb_s_rmse - xgb_tuned_rmse:+.3f}')
joblib.dump({'pre': pre_xgb, 'model': xgb_tuned, 'optuna_params': best_xgb_params},
            ADV_MODELS_DIR / 'score_xgb_tuned.pkl')

In [ ]:
%%time
# LightGBM score HPO
def lgb_score_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_est', 300, 2000),
        max_depth        = trial.suggest_int('depth', 4, 12),
        num_leaves       = trial.suggest_int('leaves', 31, 255),
        learning_rate    = trial.suggest_float('lr', 0.01, 0.3, log=True),
        subsample        = trial.suggest_float('sub', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('col', 0.5, 1.0),
        reg_lambda       = trial.suggest_float('lam', 0.1, 10.0, log=True),
        reg_alpha        = trial.suggest_float('alpha', 0.0, 5.0),
        min_child_samples= trial.suggest_int('mcs', 5, 50),
    )
    m = lgb.LGBMRegressor(**params, device=lgb_device, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(
        X_tr_lgb, y_tr_s, eval_set=[(X_va_lgb, y_va_s)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
    )
    return mean_absolute_error(y_va_s, m.predict(X_va_lgb))

study_lgb_s = optuna.create_study(direction='minimize', study_name='lgb_score')
study_lgb_s.optimize(lgb_score_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb_s.best_params
print(f'\nLGB best val MAE: {study_lgb_s.best_value:.4f}')

# Retrain best LGB on tv
lgb_tuned = lgb.LGBMRegressor(
    **{k.replace('n_est','n_estimators').replace('depth','max_depth').replace('leaves','num_leaves')
       .replace('sub','subsample').replace('col','colsample_bytree').replace('lam','reg_lambda')
       .replace('alpha','reg_alpha').replace('mcs','min_child_samples').replace('lr','learning_rate'): v
       for k, v in best_lgb_params.items()},
    device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_tuned.fit(X_tv_lgb, y_tv_s)
lgb_tuned_mae  = mean_absolute_error(y_te_s, lgb_tuned.predict(X_te_lgb))
lgb_tuned_rmse = mean_squared_error(y_te_s, lgb_tuned.predict(X_te_lgb))**0.5
print(f'[LGB Tuned Score] MAE={lgb_tuned_mae:.3f}  RMSE={lgb_tuned_rmse:.3f}')
joblib.dump({'pre': pre_lgb, 'model': lgb_tuned, 'optuna_params': best_lgb_params},
            ADV_MODELS_DIR / 'score_lgb_tuned.pkl')

In [ ]:
# Optuna optimization history plot
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vals_xgb = [t.value for t in study_xgb_s.trials if t.value is not None]
axes[0].plot(vals_xgb, alpha=0.7, color='steelblue', label='trial')
axes[0].plot(np.minimum.accumulate(vals_xgb), color='red', lw=2, label='best so far')
axes[0].set_title('XGBoost Score — Optuna MAE')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val MAE'); axes[0].legend()

vals_lgb = [t.value for t in study_lgb_s.trials if t.value is not None]
axes[1].plot(vals_lgb, alpha=0.7, color='coral', label='trial')
axes[1].plot(np.minimum.accumulate(vals_lgb), color='red', lw=2, label='best so far')
axes[1].set_title('LightGBM Score — Optuna MAE')
axes[1].set_xlabel('Trial'); axes[1].set_ylabel('Val MAE'); axes[1].legend()

plt.tight_layout(); plt.show()

## 🧠 Step 12 — PyTorch TabTransformer (Attention-based DL)

In [ ]:
if SKIP_DL:
    print('SKIP_DL=True — skipping TabTransformer.')
else:
    print('TabTransformer training will start in the next cell.')

In [ ]:
%%time
if not SKIP_DL:
    from model_zoo import TorchTabTransformerRegressor, TorchTabTransformerClassifier

    SCORE_CAT = [f for f in CAT_FEATS if f in X_tr_s.columns]
    SCORE_NUM = [f for f in ALL_FEATS if f not in CAT_FEATS and f in X_tr_s.columns]

    print(f'Training TabTransformer (Score) on {DEVICE} ...')
    tt_score = TorchTabTransformerRegressor(
        cat_cols=SCORE_CAT, num_cols=SCORE_NUM,
        embed_dim=32, n_heads=4, n_layers=3,
        hidden_dims=(512, 256, 128),
        dropout=0.1, batch_size=4096,
        epochs=100, lr=8e-4, patience=12, seed=42,
    )
    tt_score.fit(X_tr_s, y_tr_s, X_va_s, y_va_s, device=DEVICE)
    tt_s_preds = tt_score.predict(X_te_s, device='cpu')
    tt_s_mae   = mean_absolute_error(y_te_s, tt_s_preds)
    tt_s_rmse  = mean_squared_error(y_te_s, tt_s_preds)**0.5
    print(f'[TabTransformer Score] MAE={tt_s_mae:.3f}  RMSE={tt_s_rmse:.3f}')
    joblib.dump(tt_score, ADV_MODELS_DIR / 'score_tabtransformer.pkl')

In [ ]:
%%time
if not SKIP_DL:
    WIN_CAT = [f for f in CAT_FEATS if f in X_tr_w.columns]
    WIN_NUM = [f for f in ALL_FEATS if f not in CAT_FEATS and f in X_tr_w.columns]

    print(f'Training TabTransformer (Win) on {DEVICE} ...')
    tt_win = TorchTabTransformerClassifier(
        cat_cols=WIN_CAT, num_cols=WIN_NUM,
        embed_dim=32, n_heads=4, n_layers=3,
        hidden_dims=(512, 256, 128),
        dropout=0.1, batch_size=4096,
        epochs=100, lr=8e-4, patience=12, seed=42,
    )
    tt_win.fit(X_tr_w, y_tr_w, X_va_w, y_va_w, device=DEVICE)
    tt_w_probs = tt_win.predict_proba(X_te_w)[:, 1]
    tt_w_acc   = accuracy_score(y_te_w, (tt_w_probs>=0.5).astype(int))
    tt_w_ll    = log_loss(y_te_w, np.clip(tt_w_probs, 1e-7, 1-1e-7))
    print(f'[TabTransformer Win]   Acc={tt_w_acc:.4f}  LogLoss={tt_w_ll:.4f}')
    joblib.dump(tt_win, ADV_MODELS_DIR / 'win_tabtransformer.pkl')

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(tt_score.val_history_, color='steelblue')
    axes[0].set_title('TabTransformer Score — Val RMSE per Epoch')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE')
    axes[1].plot(tt_win.val_history_, color='coral')
    axes[1].set_title('TabTransformer Win — Val LogLoss per Epoch')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LogLoss')
    plt.tight_layout(); plt.show()

## 🔢 Step 13 — Phase-Specific Models (NEW)

In [ ]:
%%time
from model_zoo import PhaseModelBundle
import lightgbm as lgb

def make_lgb_reg():
    pre = lgb_preprocessor()
    class _Wrapped:
        def fit(self, X, y):
            self._pre = lgb_preprocessor()
            self._m = lgb.LGBMRegressor(
                n_estimators=500, max_depth=7, num_leaves=63,
                learning_rate=0.07, subsample=0.8, colsample_bytree=0.8,
                device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
            )
            self._m.fit(self._pre.fit_transform(X), y)
        def predict(self, X):
            return self._m.predict(self._pre.transform(X))
    return _Wrapped()

print('Training phase-specific score models (powerplay / middle / death) ...')
X_tr_s_ph = X_tr_s.copy()
X_va_s_ph = X_va_s.copy()

phase_bundle = PhaseModelBundle(make_lgb_reg)
phase_bundle.fit(X_tr_s_ph, y_tr_s, X_va_s_ph, y_va_s)

# Evaluate on test
X_te_s_ph = X_te_s.copy()
phase_preds = phase_bundle.predict(X_te_s_ph)
valid_ph = ~np.isnan(phase_preds)
ph_mae  = mean_absolute_error(y_te_s[valid_ph], phase_preds[valid_ph])
ph_rmse = mean_squared_error(y_te_s[valid_ph], phase_preds[valid_ph])**0.5
print(f'\n[Phase Bundle] Test MAE={ph_mae:.3f}  RMSE={ph_rmse:.3f}')
joblib.dump(phase_bundle, ADV_MODELS_DIR / 'score_phase_bundle.pkl')

## 🏗️ Step 14 — Stacking Ensemble (Level-2 Meta-Learner)

In [ ]:
%%time
if SKIP_STACKING:
    print('SKIP_STACKING=True — skipping.')
else:
    from model_zoo import StackingRegressor, StackingClassifier
    import lightgbm as lgb

    # Build wrapped models for stacking (all using pre-encoded features)
    class XGBWrapped:
        def fit(self, X, y):
            self._p = tree_preprocessor()
            self._m = xgb.XGBRegressor(
                n_estimators=500, max_depth=7, learning_rate=0.07,
                subsample=0.8, colsample_bytree=0.8, device=xgb_device,
                tree_method='hist', random_state=42, n_jobs=-1,
            )
            self._m.fit(self._p.fit_transform(X), y)
        def predict(self, X):
            return self._m.predict(self._p.transform(X))

    class LGBWrapped:
        def fit(self, X, y):
            self._p = lgb_preprocessor()
            self._m = lgb.LGBMRegressor(
                n_estimators=500, max_depth=7, learning_rate=0.07,
                subsample=0.8, colsample_bytree=0.8,
                device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
            )
            self._m.fit(self._p.fit_transform(X), y)
        def predict(self, X):
            return self._m.predict(self._p.transform(X))

    class CBWrapped:
        def fit(self, X, y):
            self._m = CatBoostRegressor(
                iterations=400, depth=7, learning_rate=0.07,
                l2_leaf_reg=2, cat_features=CAT_FEATS,
                task_type=cb_task, verbose=0, random_seed=42,
            )
            Xp = X.copy(); Xp[CAT_FEATS] = Xp[CAT_FEATS].fillna('Unknown').astype(str)
            self._m.fit(Xp, y)
        def predict(self, X):
            Xp = X.copy(); Xp[CAT_FEATS] = Xp[CAT_FEATS].fillna('Unknown').astype(str)
            return self._m.predict(Xp)

    class HGBWrapped:
        def fit(self, X, y):
            self._p = hgb_preprocessor()
            self._m = HistGradientBoostingRegressor(
                max_iter=400, max_depth=8, learning_rate=0.07, random_state=42
            )
            self._m.fit(self._p.fit_transform(X), y)
        def predict(self, X):
            return self._m.predict(self._p.transform(X))

    print('Training Stacking Regressor (5-fold OOF → Ridge meta) ...')
    stacker_s = StackingRegressor(
        base_models=[XGBWrapped(), LGBWrapped(), CBWrapped(), HGBWrapped()],
        base_names=['XGBoost', 'LightGBM', 'CatBoost', 'HGB'],
        n_folds=5, meta_alpha=0.5,
    )
    stacker_s.fit(X_tv_s, y_tv_s, X_te_s, y_te_s)
    joblib.dump(stacker_s, ADV_MODELS_DIR / 'score_stacker.pkl')

In [ ]:
%%time
if not SKIP_STACKING:
    from sklearn.calibration import CalibratedClassifierCV

    class XGBWinWrapped:
        def fit(self, X, y):
            self._p = tree_preprocessor()
            self._m = xgb.XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.07,
                subsample=0.8, colsample_bytree=0.8, device=xgb_device,
                tree_method='hist', random_state=42, n_jobs=-1,
            )
            self._m.fit(self._p.fit_transform(X), y)
        def predict_proba(self, X):
            return self._m.predict_proba(self._p.transform(X))
        def predict(self, X):
            return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    class LGBWinWrapped:
        def fit(self, X, y):
            self._p = lgb_preprocessor()
            self._m = lgb.LGBMClassifier(
                n_estimators=500, max_depth=7, learning_rate=0.07,
                subsample=0.8, colsample_bytree=0.8,
                device=lgb_device, random_state=42, n_jobs=-1, verbose=-1,
            )
            self._m.fit(self._p.fit_transform(X), y)
        def predict_proba(self, X):
            return self._m.predict_proba(self._p.transform(X))
        def predict(self, X):
            return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    class CBWinWrapped:
        def fit(self, X, y):
            self._m = CatBoostClassifier(
                iterations=400, depth=7, learning_rate=0.07,
                l2_leaf_reg=2, cat_features=CAT_FEATS,
                task_type=cb_task, verbose=0, random_seed=42,
            )
            Xp = X.copy(); Xp[CAT_FEATS] = Xp[CAT_FEATS].fillna('Unknown').astype(str)
            self._m.fit(Xp, y)
        def predict_proba(self, X):
            Xp = X.copy(); Xp[CAT_FEATS] = Xp[CAT_FEATS].fillna('Unknown').astype(str)
            return self._m.predict_proba(Xp)
        def predict(self, X):
            return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    class HGBWinWrapped:
        def fit(self, X, y):
            self._p = hgb_preprocessor()
            base = Pipeline([('pre', self._p),
                             ('m', HistGradientBoostingClassifier(
                                 max_iter=400, max_depth=6, learning_rate=0.07, random_state=42))])
            self._m = CalibratedClassifierCV(base, method='isotonic', cv=3)
            self._m.fit(X, y)
        def predict_proba(self, X):
            return self._m.predict_proba(X)
        def predict(self, X):
            return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    print('Training Stacking Classifier (5-fold OOF → Logistic meta) ...')
    stacker_w = StackingClassifier(
        base_models=[XGBWinWrapped(), LGBWinWrapped(), CBWinWrapped(), HGBWinWrapped()],
        base_names=['XGBoost', 'LightGBM', 'CatBoost', 'HGB'],
        n_folds=5, meta_C=1.0,
    )
    stacker_w.fit(X_tv_w, y_tv_w, X_te_w, y_te_w)
    joblib.dump(stacker_w, ADV_MODELS_DIR / 'win_stacker.pkl')

## 📊 Step 15 — Full Model Comparison

In [ ]:
import pandas as pd

score_results = [
    {'model': 'HGB',            'mae': hgb_s_mae,   'rmse': hgb_s_rmse},
    {'model': 'XGBoost',        'mae': xgb_s_mae,   'rmse': xgb_s_rmse},
    {'model': 'XGBoost-Tuned',  'mae': xgb_tuned_mae, 'rmse': xgb_tuned_rmse},
    {'model': 'LightGBM',       'mae': lgb_s_mae,   'rmse': lgb_s_rmse},
    {'model': 'LightGBM-Tuned', 'mae': lgb_tuned_mae, 'rmse': lgb_tuned_rmse},
    {'model': 'CatBoost',       'mae': cb_s_mae,    'rmse': cb_s_rmse},
    {'model': 'PhaseBundle',    'mae': ph_mae,       'rmse': ph_rmse},
]
if not SKIP_DL:
    score_results.append({'model': 'TabTransformer', 'mae': tt_s_mae, 'rmse': tt_s_rmse})
if not SKIP_STACKING:
    score_results.append({'model': 'Stacking', 'mae': stacker_s.test_score_['mae'], 'rmse': stacker_s.test_score_['rmse']})

score_df = pd.DataFrame(score_results).sort_values('rmse').reset_index(drop=True)
score_df[['mae','rmse']] = score_df[['mae','rmse']].round(3)

print('=== SCORE MODEL COMPARISON ===')
display(score_df.style.background_gradient(subset=['rmse'], cmap='RdYlGn_r').format({'mae':'{:.3f}','rmse':'{:.3f}'}))

In [ ]:
win_results = [
    {'model': 'HGB',          'accuracy': hgb_w_acc,  'log_loss': hgb_w_ll},
    {'model': 'XGBoost',      'accuracy': xgb_w_acc,  'log_loss': xgb_w_ll},
    {'model': 'LightGBM',     'accuracy': lgb_w_acc,  'log_loss': lgb_w_ll},
    {'model': 'CatBoost',     'accuracy': cb_w_acc,   'log_loss': cb_w_ll},
]
if not SKIP_DL:
    win_results.append({'model': 'TabTransformer', 'accuracy': tt_w_acc, 'log_loss': tt_w_ll})
if not SKIP_STACKING:
    win_results.append({'model': 'Stacking', 'accuracy': stacker_w.test_score_['accuracy'], 'log_loss': stacker_w.test_score_['log_loss']})

win_df = pd.DataFrame(win_results).sort_values('log_loss').reset_index(drop=True)
win_df[['accuracy','log_loss']] = win_df[['accuracy','log_loss']].round(4)

print('=== WIN MODEL COMPARISON ===')
display(win_df.style.background_gradient(subset=['log_loss'], cmap='RdYlGn_r').format({'accuracy':'{:.4f}','log_loss':'{:.4f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sd = score_df.set_index('model')
colors_s = ['#2ecc71' if s == sd['rmse'].min() else '#3498db' for s in sd['rmse']]
sd['rmse'].plot(kind='barh', ax=axes[0], color=colors_s)
axes[0].set_title('Score Models — Test RMSE (lower = better)')
axes[0].set_xlabel('RMSE (runs)'); axes[0].invert_yaxis()
axes[0].axvline(sd['rmse'].min(), color='green', linestyle='--', alpha=0.5)

wd = win_df.set_index('model')
colors_w = ['#2ecc71' if l == wd['log_loss'].min() else '#e74c3c' for l in wd['log_loss']]
wd['log_loss'].plot(kind='barh', ax=axes[1], color=colors_w)
axes[1].set_title('Win Models — Test LogLoss (lower = better)')
axes[1].set_xlabel('Log-Loss'); axes[1].invert_yaxis()
axes[1].axvline(wd['log_loss'].min(), color='green', linestyle='--', alpha=0.5)

plt.suptitle('Advanced Model Comparison — Green = Best', fontsize=13)
plt.tight_layout(); plt.show()

## 🔭 Step 16 — Promote Best Advanced Models to Production

In [ ]:
import shutil, json

# Best score model
best_score_name = score_df.iloc[0]['model']
best_score_file_map = {
    'HGB':            ADV_MODELS_DIR / 'score_hgb_adv.pkl',
    'XGBoost':        ADV_MODELS_DIR / 'score_xgb_adv.pkl',
    'XGBoost-Tuned':  ADV_MODELS_DIR / 'score_xgb_tuned.pkl',
    'LightGBM':       ADV_MODELS_DIR / 'score_lgb_adv.pkl',
    'LightGBM-Tuned': ADV_MODELS_DIR / 'score_lgb_tuned.pkl',
    'CatBoost':       ADV_MODELS_DIR / 'score_cb_adv.pkl',
    'Stacking':       ADV_MODELS_DIR / 'score_stacker.pkl',
    'TabTransformer': ADV_MODELS_DIR / 'score_tabtransformer.pkl',
}
best_win_name = win_df.iloc[0]['model']
best_win_file_map = {
    'HGB':            ADV_MODELS_DIR / 'win_hgb_adv.pkl',
    'XGBoost':        ADV_MODELS_DIR / 'win_xgb_adv.pkl',
    'LightGBM':       ADV_MODELS_DIR / 'win_lgb_adv.pkl',
    'CatBoost':       ADV_MODELS_DIR / 'win_cb_adv.pkl',
    'Stacking':       ADV_MODELS_DIR / 'win_stacker.pkl',
    'TabTransformer': ADV_MODELS_DIR / 'win_tabtransformer.pkl',
}

sf = best_score_file_map.get(best_score_name)
wf = best_win_file_map.get(best_win_name)

if sf and sf.exists():
    shutil.copy(sf, PROJECT_DIR / 'models' / 'score_model.pkl')
    print(f'✅ Promoted score model : {best_score_name}  (RMSE={score_df.iloc[0]["rmse"]:.3f})')
if wf and wf.exists():
    shutil.copy(wf, PROJECT_DIR / 'models' / 'win_model.pkl')
    print(f'✅ Promoted win model   : {best_win_name}  (Acc={win_df.iloc[0]["accuracy"]:.4f})')

report = {
    'trained_at': pd.Timestamp.now().isoformat(),
    'promoted': {'score': best_score_name, 'win': best_win_name},
    'score_models': score_df.to_dict('records'),
    'win_models': win_df.to_dict('records'),
    'feature_count': len(ALL_FEATS),
    'advanced_features': ADVANCED_NUMERIC_EXTRAS,
    'optuna_trials': OPTUNA_TRIALS,
}
(PROJECT_DIR / 'models' / 'advanced_training_report.json').write_text(
    json.dumps(report, indent=2)
)
print('\nReport saved to models/advanced_training_report.json')

## 🏏 Step 17 — Predict ALL Upcoming IPL 2026 Matches (One by One)

In [ ]:
from tournament import MatchPredictor, UPCOMING_2026, predict_all_upcoming, EloSystem
from feature_engine import EloSystem as EloSys

# Use the ELO system already fit on the full data
print('=== CURRENT IPL 2026 ELO STANDINGS ===')
print(elo.ratings_df().to_string(index=False))
print()

In [ ]:
# Build the MatchPredictor using the live ELO + historical features
predictor = MatchPredictor(
    elo=elo,
    df_features=df,
)

print('=' * 58)
print('  IPL 2026 — UPCOMING MATCH PREDICTIONS')
print('=' * 58)
results_df = predict_all_upcoming(predictor, UPCOMING_2026)
print(f'\n{len(results_df)} matches predicted.')

In [ ]:
# Summary table of all predictions
if not results_df.empty:
    summary = results_df[[
        'date', 'match', 'elo_win_p1', 'elo_win_p2',
        'form5_t1', 'form5_t2', 'blend_win_p1', 'blend_win_p2',
        'predicted_winner', 'confidence'
    ]].copy()
    summary.columns = [
        'Date', 'Match', 'ELO% T1', 'ELO% T2',
        'Form% T1', 'Form% T2', 'Blend% T1', 'Blend% T2',
        'Predicted Winner', 'Confidence'
    ]
    print('=== PREDICTION SUMMARY ===')
    display(summary.style
        .background_gradient(subset=['Blend% T1'], cmap='RdYlGn')
        .format({'Blend% T1': '{:.1f}', 'Blend% T2': '{:.1f}', 'Confidence': '{:.1f}%'})
    )

## 🎲 Step 18 — Tournament Simulation (10,000 runs Monte Carlo)

In [ ]:
from tournament import TournamentSimulator

# Current IPL 2026 points — UPDATE THIS with actual standings!
# Format: team -> points
current_points = {
    'Chennai Super Kings':         0,
    'Delhi Capitals':              0,
    'Gujarat Titans':              0,
    'Kolkata Knight Riders':       0,
    'Lucknow Super Giants':        0,
    'Mumbai Indians':              0,
    'Punjab Kings':                0,
    'Rajasthan Royals':            0,
    'Royal Challengers Bengaluru': 0,
    'Sunrisers Hyderabad':         0,
}

simulator = TournamentSimulator(
    predictor=predictor,
    current_points=current_points,
    n_simulations=10_000,
    seed=42,
)

print('Running 10,000 tournament simulations ...')
qual_df = simulator.simulate_all(UPCOMING_2026, top_n=4)

if not qual_df.empty:
    print('\n=== PLAYOFF QUALIFICATION PROBABILITIES (Top 4) ===')
    display(
        qual_df.style
        .background_gradient(subset=['top4_prob_pct'], cmap='RdYlGn')
        .format({'top4_prob_pct': '{:.1f}%', 'elo': '{:.0f}'})
        .set_caption('Based on 10,000 Monte Carlo simulations of remaining matches')
    )

In [ ]:
# Visualize qualification probabilities
if not qual_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar chart of qualification prob
    colors = ['#2ecc71' if p >= 50 else '#e74c3c' for p in qual_df['top4_prob_pct']]
    axes[0].barh(qual_df['short'], qual_df['top4_prob_pct'], color=colors)
    axes[0].axvline(50, color='black', linestyle='--', alpha=0.5)
    axes[0].set_title('Playoff Qualification Probability (%)')
    axes[0].set_xlabel('Probability (%)')
    axes[0].invert_yaxis()
    for i, (_, row) in enumerate(qual_df.iterrows()):
        axes[0].text(row['top4_prob_pct'] + 0.5, i, f'{row["top4_prob_pct"]:.1f}%', va='center')

    # ELO rating
    axes[1].barh(qual_df['short'], qual_df['elo'], color='steelblue')
    axes[1].axvline(1500, color='red', linestyle='--', alpha=0.5, label='Baseline')
    axes[1].set_title('Current ELO Ratings')
    axes[1].set_xlabel('ELO')
    axes[1].invert_yaxis()
    axes[1].legend()

    plt.tight_layout(); plt.show()

## 💾 Step 19 — Save Everything

In [ ]:
import zipfile, shutil
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M')
zip_path = Path(f'/content/ipl_advanced_models_{ts}.zip')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # All advanced models
    for f in ADV_MODELS_DIR.glob('*.pkl'):
        zf.write(f, f'advanced/{f.name}')
    # Promoted production models
    for fname in ['score_model.pkl', 'win_model.pkl']:
        p = PROJECT_DIR / 'models' / fname
        if p.exists():
            zf.write(p, fname)
    # Reports
    for fname in ['advanced_training_report.json', 'all_models_report.json', 'deployment_report.json']:
        p = PROJECT_DIR / 'models' / fname
        if p.exists():
            zf.write(p, fname)
    # Support tables
    proc_dir = PROJECT_DIR / 'data' / 'processed'
    for fname in ['venue_stats.csv', 'team_form_latest.csv', 'team_venue_form_latest.csv',
                  'matchup_form_latest.csv', 'active_teams_2026.csv']:
        p = proc_dir / fname
        if p.exists():
            zf.write(p, f'support/{fname}')

size_mb = zip_path.stat().st_size / 1e6
print(f'✅ Archive: {zip_path.name}  ({size_mb:.1f} MB)')

if USE_GOOGLE_DRIVE:
    dst = Path(DRIVE_SAVE_PATH) / zip_path.name
    shutil.copy(zip_path, dst)
    print(f'✅ Saved to Drive: {dst}')

In [ ]:
from google.colab import files
print(f'Downloading {zip_path.name} ...')
files.download(str(zip_path))

## 🏁 Summary

| What was trained | Where saved |
|-----------------|-------------|
| HGB score + win | `models/advanced/score_hgb_adv.pkl` |
| XGBoost score + win | `models/advanced/score_xgb_adv.pkl` |
| XGBoost Optuna-tuned | `models/advanced/score_xgb_tuned.pkl` |
| LightGBM score + win | `models/advanced/score_lgb_adv.pkl` |
| LightGBM Optuna-tuned | `models/advanced/score_lgb_tuned.pkl` |
| CatBoost score + win | `models/advanced/score_cb_adv.pkl` |
| Phase-specific bundle | `models/advanced/score_phase_bundle.pkl` |
| TabTransformer (DL) | `models/advanced/score_tabtransformer.pkl` |
| Stacking ensemble | `models/advanced/score_stacker.pkl` |
| Best → promoted | `models/score_model.pkl`, `models/win_model.pkl` |
| Comparison report | `models/advanced_training_report.json` |

**To update the upcoming matches** — edit `UPCOMING_2026` in `colab_training/tournament.py`
with the real IPL 2026 schedule and re-run Steps 17–18.

**To update current points** — edit the `current_points` dict in Step 18 with the actual IPL standings.
